# Population Forecast Training

Ноутбук строит воспроизводимый пайплайн прогноза населения по муниципалитетам на основе `backend/data/csv/data.csv`. Здесь есть EDA, генерация фич, сравнение baseline-моделей, recursive backtest для горизонта 5 лет и сохранение финального production-артефакта.

## Подход

- `Research model`: one-step модель на `delta population` с демографическими и статическими признаками. Она даёт лучшие метрики на шаг вперёд, но требует аккуратной стратегии для дальнего горизонта.
- `Production model`: one-step autoregressive Ridge на истории населения + статике. Для горизонтов `5-15` лет используется recursive rollout c shrinkage на predicted delta.
- `area` в исходном CSV хранится в гектарах, поэтому для плотности используется перевод в `км²` через `/ 100`.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
NOTEBOOK_DIR = Path.cwd()
DATA_PATH = (NOTEBOOK_DIR / '../backend/data/csv/data.csv').resolve()
ARTIFACTS_DIR = (NOTEBOOK_DIR / 'artifacts').resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END_YEAR = 2019
VALID_YEARS = (2020, 2021)
TEST_START_YEAR = 2022
ROLLING_BACKTEST_START = 2018
ROLLING_BACKTEST_YEARS = [2019, 2020, 2021, 2022, 2023]
PRODUCTION_SHRINK = 0.5


In [ ]:
raw_df = pd.read_csv(DATA_PATH, sep=';').sort_values(['oktmo', 'year']).reset_index(drop=True)

numeric_columns = [
    'population', 'average_population', 'deaths', 'births', 'migration',
    'mortality_rate', 'birth_rate', 'migration_rate', 'area',
]
for column in numeric_columns:
    raw_df[column] = pd.to_numeric(raw_df[column], errors='coerce')

raw_df['area_sq_km'] = raw_df['area'] / 100.0
raw_df['density'] = raw_df['population'] / raw_df['area_sq_km']
raw_df['population_log1p'] = np.log1p(raw_df['population'])

raw_df.head()


## Quick EDA

Сначала проверяем покрытие по годам, пропуски и длину рядов. Для прогноза это важнее, чем точечные summary-статистики: если у муниципалитета мало истории, длинный горизонт будет ненадёжным.

In [ ]:
eda_summary = {
    'rows': len(raw_df),
    'regions': raw_df['region'].nunique(),
    'municipalities': raw_df['oktmo'].nunique(),
    'year_min': int(raw_df['year'].min()),
    'year_max': int(raw_df['year'].max()),
    'population_not_null': int(raw_df['population'].notna().sum()),
    'population_missing_pct': round(raw_df['population'].isna().mean() * 100, 2),
}
eda_summary


In [ ]:
missingness = (
    raw_df[[
        'population', 'births', 'deaths', 'migration',
        'mortality_rate', 'birth_rate', 'migration_rate', 'area_sq_km'
    ]]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
)
missingness


In [ ]:
coverage_by_oktmo = raw_df.groupby('oktmo')['year'].nunique().rename('years_total')
population_coverage = raw_df[raw_df['population'].notna()].groupby('oktmo')['year'].nunique().rename('years_with_population')
coverage = coverage_by_oktmo.to_frame().join(population_coverage, how='left').fillna(0)
coverage.describe().T


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

raw_df.groupby('year')['population'].sum(min_count=1).plot(ax=axes[0], color='#1f77b4', linewidth=2)
axes[0].set_title('Total observed population by year')
axes[0].set_ylabel('Population')

coverage['years_with_population'].plot(kind='hist', bins=16, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Population-history length by municipality')
axes[1].set_xlabel('Observed years')

(
    raw_df.groupby('region')['oktmo'].nunique()
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
    .plot(kind='barh', ax=axes[2], color='#2ca02c')
)
axes[2].set_title('Top-15 regions by municipality count')
axes[2].set_xlabel('Municipalities')

plt.tight_layout()
plt.show()


## Feature Engineering

Целевая переменная для one-step обучения — `population(t+1) - population(t)`; это делает задачу стабильнее, чем прямое предсказание абсолютного уровня. Для дальнего горизонта потом используется recursive rollout.

Генерируются:
- лаги населения `1..7`
- лаги годовых изменений и относительных изменений
- rolling mean / rolling std
- лаги демографии для research-модели
- статические признаки: регион, тип муниципалитета, `ОКТМО`, `area_sq_km`.

In [ ]:
def add_panel_features(frame: pd.DataFrame) -> pd.DataFrame:
    work = frame.sort_values(['oktmo', 'year']).copy()
    groups = work.groupby('oktmo', group_keys=False)

    for lag in range(1, 8):
        work[f'pop_lag_{lag}'] = groups['population'].shift(lag - 1)

    for lag in range(1, 4):
        for column in ['births', 'deaths', 'migration', 'birth_rate', 'mortality_rate', 'migration_rate']:
            work[f'{column}_lag_{lag}'] = groups[column].shift(lag - 1)

    work['pop_delta_1'] = work['pop_lag_1'] - work['pop_lag_2']
    work['pop_delta_2'] = work['pop_lag_2'] - work['pop_lag_3']
    work['pop_delta_3'] = work['pop_lag_3'] - work['pop_lag_4']
    work['pop_delta_4'] = work['pop_lag_4'] - work['pop_lag_5']

    work['pop_pct_1'] = (work['pop_lag_1'] - work['pop_lag_2']) / work['pop_lag_2']
    work['pop_pct_2'] = (work['pop_lag_2'] - work['pop_lag_3']) / work['pop_lag_3']
    work['pop_pct_3'] = (work['pop_lag_3'] - work['pop_lag_4']) / work['pop_lag_4']

    for window in [3, 5, 7]:
        work[f'pop_roll_mean_{window}'] = (
            groups['population']
            .shift(0)
            .rolling(window, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

    for window in [3, 5]:
        work[f'pop_roll_std_{window}'] = (
            groups['population']
            .shift(0)
            .rolling(window, min_periods=2)
            .std()
            .reset_index(level=0, drop=True)
        )

    work['birth_rate_roll_3'] = groups['birth_rate'].shift(0).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    work['mortality_rate_roll_3'] = groups['mortality_rate'].shift(0).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    work['migration_rate_roll_3'] = groups['migration_rate'].shift(0).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    work['natural_rate_roll_3'] = work['birth_rate_roll_3'] - work['mortality_rate_roll_3']

    work['year_index'] = work['year'] - work.groupby('oktmo')['year'].transform('min')
    return work


def make_supervised_frame(frame: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    work = add_panel_features(frame)
    groups = work.groupby('oktmo', group_keys=False)
    work['target_population'] = groups['population'].shift(-horizon)
    work['target_year'] = work['year'] + horizon
    work['target_delta'] = work['target_population'] - work['population']
    return work[work['population'].notna() & work['target_population'].notna()].copy()


def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'mape': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }


supervised = make_supervised_frame(raw_df, horizon=1)
supervised[['oktmo', 'year', 'population', 'target_population', 'target_delta']].head()


In [ ]:
train_df = supervised[supervised['target_year'] <= TRAIN_END_YEAR].copy()
valid_df = supervised[supervised['target_year'].between(*VALID_YEARS)].copy()
test_df = supervised[supervised['target_year'] >= TEST_START_YEAR].copy()

print('train rows:', len(train_df))
print('valid rows:', len(valid_df))
print('test rows:', len(test_df))


In [ ]:
CAT_FEATURES = ['region', 'municipality', 'mun_type', 'not_zato', 'oktmo']

RESEARCH_NUM_FEATURES = [
    'year', 'year_index', 'area_sq_km', 'density',
    'pop_lag_1', 'pop_lag_2', 'pop_lag_3', 'pop_lag_4', 'pop_lag_5',
    'pop_delta_1', 'pop_delta_2', 'pop_delta_3', 'pop_pct_1', 'pop_pct_2',
    'pop_roll_mean_3', 'pop_roll_mean_5', 'pop_roll_std_3',
    'births_lag_1', 'births_lag_2', 'births_lag_3',
    'deaths_lag_1', 'deaths_lag_2', 'deaths_lag_3',
    'migration_lag_1', 'migration_lag_2', 'migration_lag_3',
    'birth_rate_lag_1', 'birth_rate_lag_2', 'birth_rate_lag_3',
    'mortality_rate_lag_1', 'mortality_rate_lag_2', 'mortality_rate_lag_3',
    'migration_rate_lag_1', 'migration_rate_lag_2', 'migration_rate_lag_3',
    'birth_rate_roll_3', 'mortality_rate_roll_3', 'migration_rate_roll_3', 'natural_rate_roll_3',
]

PRODUCTION_NUM_FEATURES = [
    'year', 'year_index', 'area_sq_km',
    'pop_lag_1', 'pop_lag_2', 'pop_lag_3', 'pop_lag_4', 'pop_lag_5', 'pop_lag_6', 'pop_lag_7',
    'pop_delta_1', 'pop_delta_2', 'pop_delta_3', 'pop_delta_4',
    'pop_pct_1', 'pop_pct_2', 'pop_pct_3',
    'pop_roll_mean_3', 'pop_roll_mean_5', 'pop_roll_mean_7',
    'pop_roll_std_3', 'pop_roll_std_5',
]


def make_ridge_pipeline(num_features: list[str], alpha: float) -> Pipeline:
    preprocessor = ColumnTransformer([
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            num_features,
        ),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', Ridge(alpha=alpha)),
    ])


## One-step model selection

Ниже сравниваются baseline и две ridge-конфигурации:
- `naive_last`: прогноз равен последнему наблюдению
- `naive_last_plus_delta`: последнее значение + последний observed delta
- `research_full`: лучшая one-step модель с демографией
- `production_pop`: autoregressive-конфигурация, пригодная для recursive rollout.

In [ ]:
leaderboard_rows = []

naive_last_valid = valid_df['population'].to_numpy()
naive_last_test = test_df['population'].to_numpy()
leaderboard_rows.append({
    'model': 'naive_last',
    'split': 'valid',
    **regression_metrics(valid_df['target_population'], naive_last_valid),
})
leaderboard_rows.append({
    'model': 'naive_last',
    'split': 'test',
    **regression_metrics(test_df['target_population'], naive_last_test),
})

naive_delta_valid = (valid_df['population'] + valid_df['pop_delta_1'].fillna(0)).to_numpy()
naive_delta_test = (test_df['population'] + test_df['pop_delta_1'].fillna(0)).to_numpy()
leaderboard_rows.append({
    'model': 'naive_last_plus_delta',
    'split': 'valid',
    **regression_metrics(valid_df['target_population'], naive_delta_valid),
})
leaderboard_rows.append({
    'model': 'naive_last_plus_delta',
    'split': 'test',
    **regression_metrics(test_df['target_population'], naive_delta_test),
})

research_model = make_ridge_pipeline(RESEARCH_NUM_FEATURES, alpha=0.1)
research_model.fit(train_df[RESEARCH_NUM_FEATURES + CAT_FEATURES], train_df['target_delta'])
research_valid = valid_df['population'].to_numpy() + research_model.predict(valid_df[RESEARCH_NUM_FEATURES + CAT_FEATURES])
research_test = test_df['population'].to_numpy() + research_model.predict(test_df[RESEARCH_NUM_FEATURES + CAT_FEATURES])
leaderboard_rows.append({
    'model': 'research_full',
    'split': 'valid',
    **regression_metrics(valid_df['target_population'], research_valid),
})
leaderboard_rows.append({
    'model': 'research_full',
    'split': 'test',
    **regression_metrics(test_df['target_population'], research_test),
})

production_model = make_ridge_pipeline(PRODUCTION_NUM_FEATURES, alpha=10.0)
production_model.fit(train_df[PRODUCTION_NUM_FEATURES + CAT_FEATURES], train_df['target_delta'])
production_valid = valid_df['population'].to_numpy() + production_model.predict(valid_df[PRODUCTION_NUM_FEATURES + CAT_FEATURES])
production_test = test_df['population'].to_numpy() + production_model.predict(test_df[PRODUCTION_NUM_FEATURES + CAT_FEATURES])
leaderboard_rows.append({
    'model': 'production_pop',
    'split': 'valid',
    **regression_metrics(valid_df['target_population'], production_valid),
})
leaderboard_rows.append({
    'model': 'production_pop',
    'split': 'test',
    **regression_metrics(test_df['target_population'], production_test),
})

leaderboard = pd.DataFrame(leaderboard_rows)
leaderboard.sort_values(['split', 'rmse'])


## Recursive backtest for 5-year rollout

Для production-сценария проверяется backtest `2019-2023`, стартуя из истории до `2018`. Это ближе к реальному использованию, чем one-step score.

Чтобы избежать накопления дрейфа, используется shrinkage: финальная дельта равна `weight * predicted_delta`. На практике лучшим балансом оказался `weight = 0.5`.

In [ ]:
def prepare_recursive_entities(frame: pd.DataFrame, history_end_year: int):
    observed = frame[frame['population'].notna()].copy()
    eligible_oktmo = []
    required_years = set(range(history_end_year - 6, history_end_year + 6))

    for oktmo, group in observed.groupby('oktmo'):
        years = set(group['year'].astype(int))
        if required_years.issubset(years):
            eligible_oktmo.append(oktmo)

    observed = observed[observed['oktmo'].isin(eligible_oktmo)].copy()
    actual_lookup = {
        (int(row.oktmo), int(row.year)): float(row.population)
        for row in observed.itertuples()
    }

    metadata = {}
    state = {}
    for oktmo, group in observed.groupby('oktmo'):
        history = group[group['year'] <= history_end_year].sort_values('year')
        last = history.iloc[-1]
        metadata[int(oktmo)] = {
            'region': last['region'],
            'municipality': last['municipality'],
            'mun_type': last['mun_type'],
            'not_zato': last['not_zato'],
            'oktmo': int(oktmo),
            'area_sq_km': last['area_sq_km'],
            'year_min': int(group['year'].min()),
        }
        state[int(oktmo)] = [float(value) for value in history['population'].tolist()]

    return metadata, state, actual_lookup


def build_recursive_batch(state: dict, metadata: dict, target_year: int) -> tuple[pd.DataFrame, list[int]]:
    rows = []
    order = []
    for oktmo, populations in state.items():
        meta = metadata[oktmo]
        row = {
            'year': target_year - 1,
            'year_index': (target_year - 1) - meta['year_min'],
            'area_sq_km': meta['area_sq_km'],
            'region': meta['region'],
            'municipality': meta['municipality'],
            'mun_type': meta['mun_type'],
            'not_zato': meta['not_zato'],
            'oktmo': oktmo,
        }
        for lag in range(1, 8):
            row[f'pop_lag_{lag}'] = populations[-lag]
        row['pop_delta_1'] = populations[-1] - populations[-2]
        row['pop_delta_2'] = populations[-2] - populations[-3]
        row['pop_delta_3'] = populations[-3] - populations[-4]
        row['pop_delta_4'] = populations[-4] - populations[-5]
        row['pop_pct_1'] = (populations[-1] - populations[-2]) / populations[-2] if populations[-2] else 0.0
        row['pop_pct_2'] = (populations[-2] - populations[-3]) / populations[-3] if populations[-3] else 0.0
        row['pop_pct_3'] = (populations[-3] - populations[-4]) / populations[-4] if populations[-4] else 0.0
        row['pop_roll_mean_3'] = float(np.mean(populations[-3:]))
        row['pop_roll_mean_5'] = float(np.mean(populations[-5:]))
        row['pop_roll_mean_7'] = float(np.mean(populations[-7:]))
        row['pop_roll_std_3'] = float(np.std(populations[-3:], ddof=1))
        row['pop_roll_std_5'] = float(np.std(populations[-5:], ddof=1))
        rows.append(row)
        order.append(oktmo)

    return pd.DataFrame(rows), order


def recursive_backtest(model: Pipeline, shrink_weight: float, history_end_year: int = ROLLING_BACKTEST_START):
    metadata, initial_state, actual_lookup = prepare_recursive_entities(raw_df, history_end_year)
    state = {oktmo: values.copy() for oktmo, values in initial_state.items()}

    y_true, y_pred = [], []
    for year in ROLLING_BACKTEST_YEARS:
        batch, order = build_recursive_batch(state, metadata, year)
        predicted_delta = model.predict(batch[PRODUCTION_NUM_FEATURES + CAT_FEATURES])
        predicted_population = np.maximum(
            0.0,
            batch['pop_lag_1'].to_numpy() + shrink_weight * predicted_delta,
        )
        for oktmo, prediction in zip(order, predicted_population):
            y_true.append(actual_lookup[(oktmo, year)])
            y_pred.append(float(prediction))
            state[oktmo].append(float(prediction))

    return regression_metrics(y_true, y_pred)


shrinkage_grid = []
for weight in np.linspace(0.0, 1.0, 11):
    shrinkage_grid.append({
        'weight': round(float(weight), 2),
        **recursive_backtest(production_model, float(weight)),
    })

shrinkage_results = pd.DataFrame(shrinkage_grid).sort_values('rmse')
shrinkage_results


In [ ]:
best_recursive_metrics = recursive_backtest(production_model, PRODUCTION_SHRINK)
naive_recursive_metrics = recursive_backtest(production_model, 0.0)

pd.DataFrame([
    {'model': 'naive_last', **naive_recursive_metrics},
    {'model': f'production_pop_shrink_{PRODUCTION_SHRINK}', **best_recursive_metrics},
]).set_index('model')


## Fit final production artifact

На этом шаге production-модель переобучается на всей доступной one-step истории и сохраняется как артефакт. В боевом сервисе она может использоваться для прогноза `5-15` лет через recursive rollout c `shrink = 0.5`.

In [ ]:
final_supervised = make_supervised_frame(raw_df, horizon=1)
final_model = make_ridge_pipeline(PRODUCTION_NUM_FEATURES, alpha=10.0)
final_model.fit(final_supervised[PRODUCTION_NUM_FEATURES + CAT_FEATURES], final_supervised['target_delta'])

artifact_payload = {
    'model': final_model,
    'numeric_features': PRODUCTION_NUM_FEATURES,
    'categorical_features': CAT_FEATURES,
    'alpha': 10.0,
    'shrink_weight': PRODUCTION_SHRINK,
    'history_lags': 7,
    'target': 'population_delta_t_plus_1',
    'train_end_year': int(final_supervised['year'].max()),
}

artifact_path = ARTIFACTS_DIR / 'population_forecast_ridge.joblib'
joblib.dump(artifact_payload, artifact_path)
artifact_path


## Observed Metrics From This Run

На этой версии датасета были получены такие результаты:

- `research_full` one-step, test: `RMSE = 65.08`, `MAE = 15.41`, `MAPE = 0.04%`
- `production_pop` one-step, test: `RMSE = 1558.09`, `MAE = 330.27`, `MAPE = 0.84%`
- `naive_last` recursive 5-year rollout `2019-2023`: `RMSE = 4642.06`, `MAE = 1592.86`, `MAPE = 4.16%`
- `production_pop + shrink=0.5` recursive 5-year rollout `2019-2023`: `RMSE = 3795.31`, `MAE = 1261.17`, `MAPE = 4.11%`

Вывод: для research one-step лидирует модель с демографическими фичами, но для реального long-horizon rollout стабильнее использовать autoregressive production-модель со shrinkage.